# 3 · Representations, targets and splits

Stage three of five. Takes the raw h5ad that [stage 1](1_data.ipynb) wrote and turns it into the
file every model reads: `..._with_targets_<score>.h5ad`, carrying both cell representations, the
response matrix, and the train/val/test assignment.

Four steps, in this order and for a reason:

| | step | writes | why it must come after the previous |
|---|---|---|---|
| **A** | `scgpt` | `obsm['X_scGPT']` | needs the raw h5ad |
| **B** | `targets` | `obsm['Y_ctrp']`, `obsm['M_ctrp']`, `uns['ctrp_drugs']` | writes into the embedding file |
| **C** | `splits` | `obs['split_ctrp']` | eligibility is derived from `M_ctrp` |
| **D** | `pca` | `obsm['X_pca']`, `obsm['X_pca_train_ctrp']` | the train-only fit needs `split_ctrp` |

**[Stage 2](2_drug_selection.ipynb) has already run and is not consumed here.** The drug panel is
applied at *training* time, in [stage 4](4a_percell_training.ipynb) — see §B for why that is a decision and
not an accident.

**All four steps run once per variant.** `VARIANTS_TO_RUN` is `('hvg5000', 'all_genes')` — R1's
decision (Selin, 12.08.2026) — and each step loops over both before the next begins, so the order
above still holds within each variant. Parameterised 13.08.2026: until then this notebook ran the
single `DEFAULT_VARIANT`, so covering R1 meant a second pass with a constant edited by hand, which is
how a variant silently goes missing from a rerun. §A is the expensive step and the reason R1 had to
be decided before R2 — scGPT embeds each variant separately, in its own venv, as a subprocess.

> ⚠️ **`OVERWRITE` is currently `True` (13.08.2026), so `scgpt`'s refuse-to-overwrite guard is
> disarmed.** It was set for R2, which is a genuine rebuild. **Setting it back to `False` is part of
> closing R2.**

> ⛔ **Nothing here has been re-run.** The 03.08.2026 freeze in [TODO](../docs/TODO.md) holds until
> Selin's review finishes. Every artifact on disk predates the code that now writes it — the
> embeddings, `X_pca`, and every run under `runs/`. This notebook defines the stage; it is not a
> record of a run.

In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.layout import DEFAULT_CTRP_SCORE, DEFAULT_VARIANT, PipelinePaths
from scripts.preprocessing import pipeline

VARIANT = DEFAULT_VARIANT
SCORE = DEFAULT_CTRP_SCORE          # auc_cc; ln_ic50_cc writes its own targets h5ad

# Machine-specific, and deliberately not in layout.py: scGPT lives in a separate checkout and
# virtualenv on whichever machine runs this, so it is not a property of the project layout.
# Set it to the interpreter that has `scgpt` installed.
SCGPT_PYTHON = '/Users/selin/PycharmProjects/scGPT/.venv/bin/python'


# ⚠️ SET TO True FOR R2 ON 13.08.2026, AND IT MUST GO BACK TO False AFTERWARDS.
#
# `convert` and `scgpt` refuse to replace an existing artifact, because everything downstream
# derives from them: a silent rebuild invalidates every representation and target while leaving them
# on disk looking current. That guard is the reason this constant exists, and True disarms it for
# every subsequent run of this notebook, not only the intended one.
#
# It is True because R2 IS the genuine rebuild -- every artifact under data/processed/ predates the
# code that now produces it. Returning it to False is part of closing R2; it is written here rather
# than only in TODO.md because this line is what a later reader will actually run.
OVERWRITE = True

# THE VARIANTS R1 DECIDED (Selin, 12.08.2026): hvg5000 AND all_genes -- not all five, not
# hvg5000 alone. hvg5000 is the default training variant and all_genes is what the report's
# full-transcriptome numbers rest on; hvg1000/2000/3000 keep their current artifacts and are
# re-embedded later as a top-up if the gene-set sweep is to be like-for-like (docs/TODO.md, R1).
#
# Parameterised 13.08.2026. Until then both drivers ran the single `DEFAULT_VARIANT`, so covering
# R1's decision meant running the notebook twice and remembering to edit a constant in between --
# which is how a variant silently goes missing from a rerun. The loop is the record of the decision.
VARIANTS_TO_RUN = ('hvg5000', 'all_genes')
PATHS = {v: PipelinePaths.build(None, v, SCORE) for v in VARIANTS_TO_RUN}

paths = PATHS[VARIANT]              # kept for anything that needs a single object
for v, p in PATHS.items():
    print(f'{v:10s} -> {p.processed_dir}')
    print(f'{"":10s}    score {p.score} -> {p.targets_h5ad.name}')
    print(f'{"":10s}    input {p.raw_h5ad.name}  '
          f'({"present" if p.raw_h5ad.exists() else "MISSING - run stage 1"})')

hvg5000    -> /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000
              score auc_cc -> SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad
              input SCP542_CCLE.h5ad  (present)
all_genes  -> /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/all_genes
              score auc_cc -> SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad
              input SCP542_CCLE.h5ad  (present)


## A · scGPT embedding

scGPT pins versions this project does not, so it lives in its own virtualenv and this step is a
**subprocess**, not an import. `SCGPT_PYTHON` above is that interpreter.

The only operation applied to expression before the model reads it is a binning of each cell into 51
bins whose edges are quantiles of **that same cell's** non-zero values. Nothing is normalised or
log-transformed on this path, and nothing needs to be: binning is a rank transform, so dividing by
library size or taking a logarithm moves the values and the bin edges together and every gene lands
in the bin it started in. Normalising before scGPT is not merely unnecessary — it is inert.

Genes outside scGPT's vocabulary are dropped from `.X` in the file this writes.
⛔ That drop is **not** clean: most of it was a symbol-matching defect rather than genuine vocabulary
coverage, repaired in the pipeline but **not yet reflected in any artifact on disk**
([Corrections](../docs/steps/corrections-and-dead-ends.md#scgpt-discarded-genes-that-are-in-its-vocabulary-under-their-current-symbols)).

⚠️ **No interactive fallback.** Without `SCGPT_PYTHON` this raises and prints the command to run by
hand — it does not prompt. A blocking prompt is invisible under `jupyter nbconvert --execute` and
would hang rather than fail. If you do run it by hand, continue at §B: re-running §A afterwards
raises, because the embedding it would write now exists.

In [2]:
# The expensive step, and the reason R1 was a decision rather than a default: this runs once per
# variant, as a subprocess in the scGPT venv.
embed_h5ad = {v: pipeline.scgpt(PATHS[v], SCGPT_PYTHON, overwrite=OVERWRITE)
              for v in VARIANTS_TO_RUN}
embed_h5ad

[scgpt] /Users/selin/PycharmProjects/scGPT/.venv/bin/python /Users/selin/PycharmProjects/OncoTox/.claude/worktrees/outputs-legacy-rename/scripts/preprocessing/gen_embeds.py --input /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE.h5ad --output /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE_scGPT_human_embeddings.h5ad --model-dir /Users/selin/Desktop/OncoTox/scGPT/scGPT_human


/Users/selin/PycharmProjects/scGPT/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/Users/selin/PycharmProjects/scGPT/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


/Users/selin/PycharmProjects/scGPT/.venv/lib/python3.12/site-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/Users/selin/PycharmProjects/scGPT/.venv/lib/python3.12/site-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)


Using MPS for embedding.
  PYTORCH_ENABLE_MPS_FALLBACK=1: aten::_nested_tensor_from_mask_left_aligned is not implemented for MPS and runs on CPU; the rest runs natively on MPS.
Seeded torch and numpy with 42.
Loading AnnData from /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE.h5ad...
Resolving gene symbols against the vocabulary...
  gene symbols: 4,632 of 5,000 rows match the vocabulary directly, 133 more via their current HGNC symbol
Writing OOV gene metadata to /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE_scGPT_human_embeddings_oov_genes.csv...
Writing OOV summary to /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE_scGPT_human_embeddings_oov_summary.json...
Running scGPT embedding on mps...
scGPT - INFO - match 4765/5000 genes in vocabulary of size 60697.


/Users/selin/PycharmProjects/scGPT/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(


...

Embedding cells: 100%|█████████▉| 836/837 [32:49<00:02,  2.36s/it]

Embedding cells: 100%|██████████| 837/837 [32:49<00:00,  2.35s/it]
/Users/selin/PycharmProjects/scGPT/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


/Users/selin/PycharmProjects/scGPT/.venv/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


Sanitizing metadata for HDF5 compatibility...
Saving to /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/all_genes/SCP542_CCLE_scGPT_human_embeddings.h5ad...
Success.


{'hvg5000': PosixPath('/Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE_scGPT_human_embeddings.h5ad'),
 'all_genes': PosixPath('/Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/all_genes/SCP542_CCLE_scGPT_human_embeddings.h5ad')}

## B · Response targets

Writes `Y_ctrp` (cell × drug), its observation mask `M_ctrp`, and the column order in
`uns['ctrp_drugs']`, for every drug CTRPv2 screened against at least `min_cell_lines` of the cell
lines that overlap SCP542.

**The drug panel does not enter here — decided 12.08.2026 (Selin).** [Stage 2](2_drug_selection.ipynb)
runs before this one and its `panel.csv` *could* be passed to `ctrp_to_h5ad --drugs`. It is not,
because the panel's whole effect is on the **model**: it sets `output_dim`, that is, how many heads
share one trunk. It says what the network is asked to predict, not what the data contains. So
`Y_ctrp` / `M_ctrp` keep the full screened catalog and the panel is applied as a column selection in
[stage 4](4a_percell_training.ipynb), via `MultiDrugDataset(drugs=…)`. Written up in
[Step 03](../docs/steps/03-model-and-training-design.md#the-drug-panel-is-a-training-time-choice-not-a-property-of-the-target-file-12082026).

Two consequences of that, neither of them the reason for it:

- **Changing the panel needs no preprocessing re-run.** One targets h5ad serves any panel, which is
  why stage 2 can rebuild the panel under the freeze while these h5ads cannot be rebuilt.
- **Filtering here would have moved the splits.** §C decides eligibility from *"this line has at
  least one observed label"* — a test taken over the *width* of `M`. Narrowing `M` from ~545 columns
  to 11 would re-evaluate it against the panel and could drop lines that are screened but not against
  a panel compound, silently redrawing `split_ctrp`.

The measure is whichever `SCORE` names. `auc_cc` is complete for every curve; `ln_ic50_cc` is
missing for ~40 % of them by construction, because an IC50 that falls well outside the measured dose
range is discarded — which is most compounds that never reach half-killing.

In [3]:
targets_h5ad = {v: pipeline.targets(PATHS[v]) for v in VARIANTS_TO_RUN}
targets_h5ad

[targets] score=auc_cc, min_cell_lines=50
Loading AnnData...


Loading CTRPv2 responses from CTRPv2.csv (score=auc_cc)...


  395,024 measurements | 886 cell lines | 545 drugs
  8,187 of 395,024 rows are exact duplicates of another row (2.1 %) and are dropped.


  Overlap with SCP542: 181 cell lines out of 198 in AnnData.
  Drug filter: 534 / 545 drugs kept (>= 50 overlapping cell lines).
Building (cell line x drug) auc_cc matrix...
Mapping 534 drugs to 53513 single cells...
  Cellosaurus accessions attached for 181 cell lines (cells: 47,440 / 53,513).
  Multi-drug summary: 47440 / 53513 cells have at least one CTRP label; mean drugs/cell = 401.9.
Sanitizing metadata to fix H5AD string/index compatibility...
Saving updated AnnData to /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad...


/Users/selin/PycharmProjects/OncoTox/.venv/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


...

  395,024 measurements | 886 cell lines | 545 drugs
  8,187 of 395,024 rows are exact duplicates of another row (2.1 %) and are dropped.
  Overlap with SCP542: 181 cell lines out of 198 in AnnData.


  Drug filter: 534 / 545 drugs kept (>= 50 overlapping cell lines).
Building (cell line x drug) auc_cc matrix...
Mapping 534 drugs to 53513 single cells...
  Cellosaurus accessions attached for 181 cell lines (cells: 47,440 / 53,513).
  Multi-drug summary: 47440 / 53513 cells have at least one CTRP label; mean drugs/cell = 401.9.
Sanitizing metadata to fix H5AD string/index compatibility...
Saving updated AnnData to /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/all_genes/SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad...


Done!


/Users/selin/PycharmProjects/OncoTox/.venv/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


{'hvg5000': PosixPath('/Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad'),
 'all_genes': PosixPath('/Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/all_genes/SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad')}

## C · Train / validation / test split

A 70/15/15 partition **of cell lines**, so every cell of a line falls on the same side. The label is
defined per (cell line × drug) and broadcast to every cell of that line, so a per-cell split would
put copies of the same label on both sides — the leak that forced this design
([Step 04](../docs/steps/04-single-task-results.md)).

**The assignment is frozen to `splits/split_ctrp.csv`, not redrawn.** Eligibility depends on which
drugs survive the filters, so redrawing would move lines between train, validation and test whenever
anything upstream changed, and results from either side of that change would look comparable while
being scored on different held-out lines. A line the file does not cover is an error, not a fresh
draw.

⚠️ `regenerate=True` redraws and overwrites it. **Every result produced before it was regenerated was
scored on different held-out lines.** That is not a repair.

In [4]:
for v in VARIANTS_TO_RUN:
    pipeline.splits(PATHS[v])

[splits] seed=42, regenerate=False
Loading /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad...


Found 181 unique cell lines with at least one CTRP drug label across 47440 cells.
  Wrote frozen split to /Users/selin/PycharmProjects/OncoTox/.claude/worktrees/outputs-legacy-rename/splits/split_ctrp.csv (181 lines, seed=42).
Cell Line Split -> train: 126, val: 27, test: 28

Final Cell Split distribution for multi-drug (split_ctrp):
split_ctrp
train         34082
val            6829
test           6529
unassigned     6073
Name: count, dtype: int64

Saving updated AnnData...


/Users/selin/PycharmProjects/OncoTox/.venv/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


Done! Leakage-free grouped splits are permanently saved.
[splits] seed=42, regenerate=False
Loading /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/all_genes/SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad...


Found 181 unique cell lines with at least one CTRP drug label across 47440 cells.
  Using frozen split from /Users/selin/PycharmProjects/OncoTox/.claude/worktrees/outputs-legacy-rename/splits/split_ctrp.csv (181 lines).
Cell Line Split -> train: 126, val: 27, test: 28

Final Cell Split distribution for multi-drug (split_ctrp):
split_ctrp
train         34082
val            6829
test           6529
unassigned     6073
Name: count, dtype: int64

Saving updated AnnData...


Done! Leakage-free grouped splits are permanently saved.


/Users/selin/PycharmProjects/OncoTox/.venv/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


## D · PCA baseline

Fitted on the **convert output** — the full HVG set from [stage 1](1_data.ipynb) — and *not* on this
file's own `.X`, from which §A has dropped out-of-vocabulary genes. Both representations therefore
rest on the same single gene filter, which is what makes the comparison one of representations
rather than of gene sets.

Two keys are written, and the difference matters:

- **`X_pca`** — fitted on every cell. Descriptive: correct for UMAPs, **wrong as model input**,
  because a held-out cell's coordinates then depend on held-out cells.
- **`X_pca_train_ctrp`** — fitted on the training lines of the frozen split alone, per-gene
  standardisation and rotation both, then applied to the held-out lines. This is what a model scored
  on that split reads.

scGPT needs no such distinction: its weights are pretrained and frozen and its binning uses only the
cell being embedded, so nothing about it is fitted on this data.

Runs last because the train-only fit needs the `split_ctrp` column §C writes.

In [5]:
for v in VARIANTS_TO_RUN:
    pipeline.pca(PATHS[v])

[pca] n_comps=512, seed=42, force=False
Loading /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad...


Computing PCA on HVG-filtered counts from /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE.h5ad...


  X_pca_train_ctrp: fitted on 34082 train cells -> shape (53513, 512).


  X_pca (all cells) computed on 5000 genes -> shape (53513, 512).
    42.1% of variance retained.
Saving updated AnnData with X_pca (targets .X unchanged)...


/Users/selin/PycharmProjects/OncoTox/.venv/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


...

Computing PCA on HVG-filtered counts from /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/all_genes/SCP542_CCLE.h5ad...


  X_pca_train_ctrp: fitted on 34082 train cells -> shape (53513, 512).


  X_pca (all cells) computed on 22722 genes -> shape (53513, 512).
    20.1% of variance retained.
Saving updated AnnData with X_pca (targets .X unchanged)...


/Users/selin/PycharmProjects/OncoTox/.venv/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


Done! You can now run baseline training.


In [6]:
import anndata as ad
import numpy as np

for variant in VARIANTS_TO_RUN:
    a = ad.read_h5ad(targets_h5ad[variant], backed='r')
    groups = a.obs['Cell_line'].astype(str).to_numpy()
    split = a.obs['split_ctrp'].astype(str).to_numpy()
    first = {ln: split[groups == ln][0] for ln in np.unique(groups)}

    print(f'--- {variant} ---')
    print(f'cells        : {a.n_obs:,}')
    print(f'obsm         : {sorted(a.obsm.keys())}')
    print(f'K (drugs)    : {len(a.uns["ctrp_drugs"])}   score={a.uns["ctrp_score"]}')
    print(f'observed     : {int(np.asarray(a.obsm["M_ctrp"]).sum()):,} (cell x drug) labels')
    for s in ('train', 'val', 'test', 'unassigned'):
        print(f'  {s:11s}: {sum(x == s for x in first.values()):3d} cell lines')
    # Carried through convert -> scgpt -> targets. 4b stage 6 reads them off this file.
    qc = [c for c in ('total_counts', 'pct_counts_mt') if c in a.obs.columns]
    print(f'UMI qc       : {qc if qc else "MISSING -- 4b stage 6 cannot run"}')
    a.file.close()

--- hvg5000 ---
cells        : 53,513
obsm         : ['M_ctrp', 'X_pca', 'X_pca_train_ctrp', 'X_scGPT', 'Y_ctrp']
K (drugs)    : 534   score=auc_cc
observed     : 21,506,818 (cell x drug) labels
  train      : 126 cell lines
  val        :  27 cell lines
  test       :  28 cell lines
  unassigned :  17 cell lines
UMI qc       : ['total_counts', 'pct_counts_mt']


--- all_genes ---
cells        : 53,513
obsm         : ['M_ctrp', 'X_pca', 'X_pca_train_ctrp', 'X_scGPT', 'Y_ctrp']
K (drugs)    : 534   score=auc_cc
observed     : 21,506,818 (cell x drug) labels
  train      : 126 cell lines
  val        :  27 cell lines
  test       :  28 cell lines
  unassigned :  17 cell lines
UMI qc       : ['total_counts', 'pct_counts_mt']
